# synT1CE — Contrast-enhanced T1CE synthesis with CFM-OT

Predict post-contrast **T1CE** MRI from non-contrast modalities (T1n, T2w, T2f) on
BraTS-MEN using flow matching / optimal transport.

- Input `x = [T1n, T2w, T2f]` is the ControlNet conditioning; target `y = T1CE`.
- Modality-dropout at training time makes the model robust to missing inputs.
- Model and training live in the repo: https://github.com/hashirama21/CFM-OT

Run the cells top to bottom. On Kaggle, run the environment-fix cells first and
**restart the runtime** when prompted, then continue.

## 0. Environment setup

In [ ]:
# Fix a torch/torchvision ABI mismatch (torchvision::nms). Run first, then RESTART.
import torch, subprocess, sys

tv = {"2.6": "0.21.0", "2.5": "0.20.0", "2.4": "0.19.0",
      "2.3": "0.18.0", "2.2": "0.17.0", "2.1": "0.16.0"}
target = tv.get(".".join(torch.__version__.split(".")[:2]))
if target:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"torchvision=={target}", "--no-deps", "--force-reinstall"], check=False)
    print(f"torchvision pinned to {target} for torch {torch.__version__}. RESTART the runtime.")
else:
    print(f"Unknown torch {torch.__version__}; pin torchvision manually.")

In [ ]:
# Fix a Pillow ABI issue (PIL._typing._Ink). Run first, then RESTART.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-cache-dir", "Pillow==10.4.0"], check=False)
print("Pillow reinstalled. RESTART the runtime.")

In [ ]:
# Install extras without disturbing Kaggle's NumPy 2.x (avoids an ABI break with pandas).
import subprocess, sys, numpy
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nibabel", "tifffile", f"numpy=={numpy.__version__}"], check=False)
print("numpy", numpy.__version__)

## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import glob
import json
import os
import random
import subprocess
import sys
import time
import warnings
import zipfile
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity as skimage_ssim
from tqdm import tqdm

import torch

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "figure.facecolor": "white"})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}  |  VRAM {p.total_memory / 1e9:.1f} GB  |  count {torch.cuda.device_count()}")

In [ ]:
@dataclass
class Config:
    # Ephemeral scratch for the .pt tensors (recreated each session).
    data_base: Path = (
        Path("/kaggle/temp") if Path("/kaggle/temp").exists() else
        Path("/content") if Path("/content").exists() else Path("/root")
    )
    # Persistent storage for checkpoints and outputs.
    persist_base: Path = (
        Path("/kaggle/working") if Path("/kaggle/working").exists() else
        Path("/content") if Path("/content").exists() else Path("/root")
    )
    data_subpath: str = "training_data/BraTS-MEN-Train"
    tensors_name: str = "BraTS_2D_Tensors"
    out_name: str = "run_synt1ce"
    # A read-only Kaggle Dataset of .pt files, if mounted (0 disk, persistent).
    kaggle_input_dir: str = ""

    # Patient-level 80/10/10 split.
    train_frac: float = 0.80
    val_frac: float = 0.10
    split_seed: int = 42
    dataset_fraction: float = 1.0  # use a fraction of each split for quick runs

    # Training knobs passed through to the Hydra config.
    max_epochs: int = 50
    batch_size: int = 16
    modality_dropout: float = 0.5

    @property
    def data_root(self) -> Path:
        return self.data_base / self.data_subpath

    @property
    def tensors_dir(self) -> Path:
        return Path(self.kaggle_input_dir) if self.kaggle_input_dir \
            else self.data_base / self.tensors_name

    @property
    def slice_index_csv(self) -> Path:
        if self.kaggle_input_dir:
            baked = self.tensors_dir / "slice_index.csv"
            return baked if baked.exists() else self.persist_base / "slice_index.csv"
        return self.tensors_dir / "slice_index.csv"

    @property
    def splits_csv(self) -> Path:
        if self.kaggle_input_dir:
            baked = self.tensors_dir / "splits.csv"
            if baked.exists():
                return baked
        return self.persist_base / "splits.csv"

    @property
    def out_dir(self) -> Path:
        return self.persist_base / "outputs" / self.out_name


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print(f"tensors_dir : {CFG.tensors_dir}")
print(f"out_dir     : {CFG.out_dir}")

## 2. Data — download the .pt tensors

In [ ]:
# Read from a mounted Kaggle Dataset if available, otherwise download the HF archive
# to a local file (avoids the streaming "Cannot seek" bug) and extract the .pt files.
import shutil, tarfile
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "krohn/synT1CE-BraTS-MEN-Tensors"
def _n_pt(d): return len(list(Path(d).glob("*.pt")))

if CFG.kaggle_input_dir and Path(CFG.kaggle_input_dir).exists():
    n = _n_pt(CFG.kaggle_input_dir)
    assert n > 0, "Mounted dataset has no .pt files — check the path."
    print(f"{n:,} .pt read directly from {CFG.kaggle_input_dir} (no download).")
else:
    CFG.tensors_dir.mkdir(parents=True, exist_ok=True)
    if _n_pt(CFG.tensors_dir) >= 60000:
        print(f"{_n_pt(CFG.tensors_dir):,} .pt already present this session.")
    else:
        tok = os.environ.get("HF_TOKEN") or None
        pct = int(CFG.dataset_fraction * 100)
        archive = f"brats_men_tensors_{pct}pct.tar.gz"
        try:
            files = list_repo_files(HF_REPO, repo_type="dataset", token=tok)
            if archive not in files:
                cands = [f for f in files if f.endswith((".tar.gz", ".tgz", ".tar"))]
                assert cands, f"No .tar(.gz) archive in {HF_REPO}: {files[:12]}"
                archive = cands[0]
        except Exception as e:
            print(f"repo listing failed ({e}); trying {archive}")
        local = hf_hub_download(HF_REPO, archive, repo_type="dataset", token=tok,
                                local_dir=str(CFG.data_base / "_hf_dl"))
        tmp = CFG.data_base / "_extract"; tmp.mkdir(parents=True, exist_ok=True)
        mode = "r:gz" if str(local).endswith((".gz", ".tgz")) else "r:"
        with tarfile.open(local, mode) as tar:
            tar.extractall(str(tmp))
        moved = 0
        for p in glob.glob(str(tmp / "**" / "*.pt"), recursive=True):
            dst = CFG.tensors_dir / Path(p).name
            if not dst.exists():
                shutil.move(p, dst); moved += 1
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.rmtree(CFG.data_base / "_hf_dl", ignore_errors=True)
        try:
            sp = hf_hub_download(HF_REPO, "splits.csv", repo_type="dataset", token=tok,
                                 local_dir=str(CFG.data_base / "_hf_dl2"))
            CFG.persist_base.mkdir(parents=True, exist_ok=True)
            (CFG.persist_base / "splits.csv").write_bytes(Path(sp).read_bytes())
            shutil.rmtree(CFG.data_base / "_hf_dl2", ignore_errors=True)
        except Exception:
            pass
        print(f"{moved:,} .pt extracted → {CFG.tensors_dir} (total {_n_pt(CFG.tensors_dir):,}).")

In [ ]:
# Check the .pt record format: each file is {"x": (3,H,W), "y": (1,H,W)}.
files = sorted(glob.glob(str(CFG.tensors_dir / "*.pt")))
assert files, "No .pt found — check CFG.tensors_dir."
rec = torch.load(files[0], map_location="cpu", weights_only=True)
assert isinstance(rec, dict) and {"x", "y"} <= set(rec), f"Unexpected record: {type(rec)}"
print("x:", tuple(rec["x"].shape), "| y:", tuple(rec["y"].shape))
present = {Path(f).name.rsplit("_z", 1)[0] for f in files}
print(f"distinct patients on disk: {len(present)}")

## 3. Splits & slice index

In [ ]:
class SplitManager:
    """Creates and loads patient-level 80/10/10 splits (reproducible via split_seed)."""

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg

    def _all_patient_ids(self) -> List[str]:
        files = list(self.cfg.tensors_dir.glob("*.pt"))
        if files:
            return sorted({f.name.rsplit("_z", 1)[0] for f in files})
        if self.cfg.data_root.exists():
            return sorted(d.name for d in self.cfg.data_root.iterdir() if d.is_dir())
        return []

    def make(self, force: bool = False) -> None:
        if self.cfg.splits_csv.exists() and not force:
            print(f"splits.csv already exists: {self.cfg.splits_csv}")
            return
        pids = self._all_patient_ids()
        if not pids:
            print("No patients found — download the tensors first.")
            return
        rng = np.random.default_rng(self.cfg.split_seed)
        pids = list(rng.permutation(pids))
        n = len(pids)
        t = int(n * self.cfg.train_frac)
        v = int(n * (self.cfg.train_frac + self.cfg.val_frac))
        rows = (
            [{"patient_id": p, "split": "train"} for p in pids[:t]]
            + [{"patient_id": p, "split": "val"} for p in pids[t:v]]
            + [{"patient_id": p, "split": "test"} for p in pids[v:]]
        )
        pd.DataFrame(rows).to_csv(self.cfg.splits_csv, index=False)
        print(f"{n} patients → {self.cfg.splits_csv}  (train {t} | val {v-t} | test {n-v})")

    def load(self) -> Dict[str, List[str]]:
        if not self.cfg.splits_csv.exists():
            self.make()
        df = pd.read_csv(self.cfg.splits_csv)
        splits = defaultdict(list)
        for _, row in df.iterrows():
            splits[row["split"]].append(row["patient_id"])
        frac = self.cfg.dataset_fraction
        if frac < 1.0:
            rng = np.random.default_rng(self.cfg.split_seed + 1)
            for key in splits:
                pids = list(rng.permutation(splits[key]))
                splits[key] = pids[:max(1, int(len(pids) * frac))]
        print("patients per split:",
              {k: len(v) for k, v in splits.items()}, f"(fraction={frac:.0%})")
        return dict(splits)


split_mgr = SplitManager(CFG)
split_mgr.make()
splits = split_mgr.load()

In [ ]:
# Build slice_index.csv with a has_tumour flag (enhancement-derived, no GT label leakage).
if CFG.slice_index_csv.exists() and CFG.kaggle_input_dir:
    idx = pd.read_csv(CFG.slice_index_csv)
    print(f"index already present: {CFG.slice_index_csv} ({len(idx):,} slices)")
else:
    threshold, min_pixels = 0.8, 200
    rows = []
    for f in tqdm(sorted(CFG.tensors_dir.glob("*.pt")), desc="index"):
        rec = torch.load(f, map_location="cpu", weights_only=True)
        pid, z = f.stem.rsplit("_z", 1)
        x, y = rec["x"][0].numpy(), rec["y"][0].numpy()
        n_vox = int(((y - x) > threshold).sum())
        rows.append({"pid": pid, "z": int(z), "file": f.name,
                     "has_tumour": int(n_vox > min_pixels),
                     "tumour_frac": float(n_vox / (240 * 240))})
    idx = pd.DataFrame(rows)
    CFG.slice_index_csv.parent.mkdir(parents=True, exist_ok=True)
    idx.to_csv(CFG.slice_index_csv, index=False)
    print(f"{len(idx):,} slices, {idx['has_tumour'].sum():,} with tumour "
          f"({idx['has_tumour'].mean():.1%}).")

In [ ]:
# Load the index and materialize per-split slice tables.
idx = pd.read_csv(CFG.slice_index_csv)
splits_df = pd.read_csv(CFG.splits_csv)

def get_split(split_name, fraction=None):
    pids = splits_df.loc[splits_df["split"] == split_name, "patient_id"].tolist()
    if fraction and fraction < 1.0:
        rng = np.random.default_rng(42)
        pids = list(rng.permutation(sorted(pids)))[:max(1, int(len(pids) * fraction))]
    return idx[idx["pid"].isin(set(pids))].reset_index(drop=True)

f = CFG.dataset_fraction
train_idx = get_split("train", f)
val_idx = get_split("val", f)
test_idx = get_split("test", f)
print(f"train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,} slices")

## 4. Metrics suite

Enhancement-region metrics (EVarΔ, Wasserstein-1, multimodality) plus SSIM/PSNR.

In [ ]:
from scipy.stats import wasserstein_distance


def enhancement_mask(t1ce, t1n, threshold=0.1):
    """Intensity-derived enhancement region (no BraTS label, safe at inference)."""
    return (t1ce - t1n) > threshold


def compute_evar_delta(test_idx, pred_dir, tensors_dir, threshold=0.1):
    """EVarΔ(p) = Var_z(mean_enh_pred) / Var_z(mean_enh_gt); 1 = perfect coherence."""
    per_patient = {}
    for pid, group in test_idx[test_idx["has_tumour"] == True].groupby("pid"):
        group = group.sort_values("z")
        dp, dg = [], []
        for _, row in group.iterrows():
            npy = pred_dir / str(pid) / (Path(row["file"]).stem + ".npy")
            if not npy.exists():
                continue
            pred = np.load(npy)
            rec = torch.load(tensors_dir / row["file"], map_location="cpu", weights_only=True)
            t1n, t1ce = rec["x"][0].numpy(), rec["y"][0].numpy()
            m = enhancement_mask(t1ce, t1n, threshold)
            if m.sum() < 10:
                continue
            dp.append(float((pred - t1n)[m].mean()))
            dg.append(float((t1ce - t1n)[m].mean()))
        if len(dg) < 2:
            continue
        vg = float(np.var(dg, ddof=1))
        if vg < 1e-8:
            continue
        per_patient[pid] = float(np.var(dp, ddof=1)) / vg
    vals = list(per_patient.values())
    if not vals:
        return {"per_patient": {}, "mean": None, "std": None, "n": 0}
    return {"per_patient": per_patient, "mean": float(np.mean(vals)),
            "std": float(np.std(vals)), "median": float(np.median(vals)),
            "pct_lt1": float(np.mean(np.array(vals) < 1.0) * 100), "n": len(vals)}


def prove_multimodality(train_idx, tensors_dir, n_pairs=200, epsilon=0.01, threshold=0.1):
    """Empirical p(T1CE|x) multimodality: matched inputs with high enhancement L1 gap."""
    tumour_pids = list(train_idx[train_idx["has_tumour"] == True]["pid"].unique())
    rng = np.random.default_rng(42)
    matched_diffs, all_msd = [], []
    attempts = matched = 0
    while matched < n_pairs and attempts < n_pairs * 20:
        attempts += 1
        pid_i, pid_j = rng.choice(tumour_pids, size=2, replace=False)
        rows_i = train_idx[(train_idx["pid"] == pid_i) & (train_idx["has_tumour"] == True)]
        rows_j = train_idx[(train_idx["pid"] == pid_j) & (train_idx["has_tumour"] == True)]
        if rows_i.empty or rows_j.empty:
            continue
        row_i = rows_i.sample(1, random_state=int(attempts)).iloc[0]
        row_j = rows_j.sample(1, random_state=int(attempts) + 1).iloc[0]
        try:
            rec_i = torch.load(tensors_dir / row_i["file"], map_location="cpu", weights_only=True)
            rec_j = torch.load(tensors_dir / row_j["file"], map_location="cpu", weights_only=True)
        except Exception:
            continue
        x_i, y_i = rec_i["x"].numpy(), rec_i["y"][0].numpy()
        x_j, y_j = rec_j["x"].numpy(), rec_j["y"][0].numpy()
        msd = float(np.mean((x_i - x_j) ** 2))
        all_msd.append(msd)
        if msd > epsilon:
            continue
        matched += 1
        t1n_i, t1n_j = x_i[0], x_j[0]
        common = enhancement_mask(y_i, t1n_i, threshold) | enhancement_mask(y_j, t1n_j, threshold)
        if common.sum() < 5:
            continue
        matched_diffs.append(float(np.mean(np.abs(
            (y_i - t1n_i)[common] - (y_j - t1n_j)[common]))))
    return {"n_matched_pairs": len(matched_diffs), "n_attempts": attempts,
            "matched_L1_mean": float(np.mean(matched_diffs)) if matched_diffs else None,
            "matched_L1_std": float(np.std(matched_diffs)) if matched_diffs else None}


print("metrics suite ready.")

## 5. Clone CFM-OT & install

In [ ]:
# Clone a fresh copy of the repo (always latest) and install it. Version floors
# reuse the platform's torch/numpy and only add hydra/omegaconf/flow_matching/monai.
import shutil
if os.path.isdir("CFM-OT"):
    shutil.rmtree("CFM-OT")
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/hashirama21/CFM-OT.git"], check=True)
%cd CFM-OT
%pip install -q -e .
%cd ..
print("CFM-OT installed.")

In [ ]:
# Shared Hydra overrides (brats_synt1ce experiment pointed at our data/paths) + param count.
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

REPO = Path("CFM-OT").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from utils.config_schema import register_configs
register_configs()
from utils.utils_fm import build_model

CKPT_DIR = str(CFG.out_dir / "checkpoints")

# Hardware-agnostic overrides: the `hpc` train profile (used by brats_synt1ce)
# auto-selects accelerator/devices/strategy/precision, so we only pass data paths
# and run knobs. Works unchanged on 1 GPU, 2xT4, A100, H100/H200, or CPU.
OVERRIDES = [
    "experiment=brats_synt1ce",
    f"data_args.tensors_dir={CFG.tensors_dir}",
    f"data_args.slice_index_csv={CFG.slice_index_csv}",
    f"data_args.splits_csv={CFG.splits_csv}",
    f"data_args.modality_dropout={CFG.modality_dropout}",
    f"data_args.fraction={CFG.dataset_fraction}",
    f"data_args.fraction_seed={CFG.split_seed}",
    f"train_args.checkpoint_dir={CKPT_DIR}",
    f"train_args.num_epochs={CFG.max_epochs}",
    f"train_args.batch_size={CFG.batch_size}",
]

with initialize_config_dir(version_base=None, config_dir=str(REPO / "conf")):
    _cfg = compose(config_name="config", overrides=OVERRIDES)
_m = build_model(OmegaConf.to_container(_cfg.model_args, resolve=True), device=torch.device("cpu"))
print(f"Model: {sum(p.numel() for p in _m.parameters()) / 1e6:.2f}M params  |  "
      f"GPUs: {torch.cuda.device_count()}")
del _m

In [ ]:
# Sanity-check that the index is consistent with the .pt files (ready for lazy loading).
_idx = pd.read_csv(CFG.slice_index_csv)
_spl = pd.read_csv(CFG.splits_csv)
_pid = "patient_id" if "patient_id" in _spl.columns else _spl.columns[0]
print(f"slices: {len(_idx):,} | patients: {_spl[_pid].nunique()}")
for s in ("train", "val", "test"):
    pids = set(_spl.loc[_spl["split"] == s, _pid])
    print(f"  {s:5s}: {int(_idx['pid'].isin(pids).sum()):,} slices")
_f = _idx["file"].iloc[random.randint(0, len(_idx) - 1)]
assert (CFG.tensors_dir / _f).exists(), f"missing .pt: {_f}"
print("index consistent — ready for lazy loading.")

## 6. Training

In [ ]:
# Launch trainer.py with the shared overrides. Training resumes automatically from the
# last checkpoint under CKPT_DIR/<run_name>, so a re-run continues where it left off.
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env.setdefault("OMP_NUM_THREADS", "1")

t0 = time.time()
rc = subprocess.run(["python", "trainer.py", *OVERRIDES], cwd="CFM-OT", env=env).returncode
print(f"exit={rc}  elapsed={(time.time() - t0) / 3600:.2f}h")

## 7. Load the trained model

In [ ]:
# Compose the config on the test split, then load the latest checkpoint (EMA weights).
from trainer import FlowMatchingLightningModule, FlowMatchingDataModule
from utils.utils_fm import sample_batch
from utils.data_lazy import resolve_split_files

# Rebuild test_idx from the SAME source of truth as the loader so predictions,
# ground truth and metadata stay aligned (even when data_args.fraction < 1.0).
_test_files, _ = resolve_split_files(str(CFG.slice_index_csv), str(CFG.splits_csv), "test",
                                     fraction=CFG.dataset_fraction, fraction_seed=CFG.split_seed)
test_idx = idx[idx["file"].isin(set(_test_files))].reset_index(drop=True)
print("test slices (aligned):", len(test_idx))

NFE = 100  # integration steps at inference (FAST-DDPM style minimum)

with initialize_config_dir(version_base=None, config_dir=str(REPO / "conf")):
    cfg = compose(config_name="config", overrides=OVERRIDES + [
        "data_args.split_val=test",
        f"solver_args.time_points={NFE}", f"solver_args.step_size={1.0 / NFE}",
        "train_args.batch_size=8", "train_args.num_workers=2",
    ])
config = OmegaConf.to_container(cfg, resolve=True)

ckpts = sorted(glob.glob(str(Path(CKPT_DIR) / "**" / "*.ckpt"), recursive=True),
               key=lambda p: Path(p).stat().st_mtime)
assert ckpts, f"No checkpoint under {CKPT_DIR}"
BEST_CKPT = ckpts[-1]

ck = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
sd = ck["state_dict"]
if any(k.startswith("_orig_mod.") for k in sd):  # strip torch.compile prefix
    sd = {k.replace("_orig_mod.", "", 1): v for k, v in sd.items()}
model = FlowMatchingLightningModule(config)
model.load_state_dict(sd, strict=True)
model.to(DEVICE).eval()
MC = config["model_args"]
SOLVER = {"method": "midpoint", "time_points": NFE, "step_size": 1.0 / NFE}
print("loaded:", BEST_CKPT)

## 8. Modality-dropout evaluation

Test all 7 modality combinations to show the model uses each input.

In [ ]:
from PIL import Image
from skimage.metrics import structural_similarity as sk_ssim

OUT = CFG.out_dir / "modality_dropout_eval"
(OUT / "png").mkdir(parents=True, exist_ok=True)
cand = sorted(glob.glob(str(CFG.tensors_dir / "*.pt")))[:16]
assert cand, "no .pt found"

def _mm(t):
    t = t.float()
    return (t - t.amin()) / (t.amax() - t.amin()).clamp_min(1e-6)

X = torch.stack([_mm(torch.load(f, weights_only=True)["x"].float()) for f in cand])
Y = torch.stack([_mm(torch.load(f, weights_only=True)["y"].float()) for f in cand])
names = ["t1n", "t2w", "t2f"]
combos = {"all_3": [0, 1, 2], "drop_t1n": [1, 2], "drop_t2w": [0, 2], "drop_t2f": [0, 1],
          "only_t1n": [0], "only_t2w": [1], "only_t2f": [2]}

def n01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (np.ptp(a) + 1e-8)

results = {}
for name, keep in combos.items():
    Xd = torch.zeros_like(X)
    for c in keep:
        Xd[:, c] = X[:, c]
    batch = {"images": Y, "masks": Xd,
             "classes": torch.zeros(len(X), 2).scatter_(1, torch.ones(len(X), 1).long(), 1.0)}
    torch.manual_seed(0)
    with torch.no_grad():
        out = sample_batch(model.model, SOLVER, batch, torch.device(DEVICE),
                           class_conditioning=bool(MC.get("with_conditioning", False)),
                           mask_conditioning=bool(MC.get("mask_conditioning", False)))
    p = out.detach().float().cpu().numpy()[:, 0]
    g = Y.numpy()[:, 0]
    ss = [float(sk_ssim(n01(p[i]), n01(g[i]), data_range=1.0)) for i in range(len(p))]
    results[name] = {"ssim_mean": float(np.mean(ss)), "ssim_std": float(np.std(ss)),
                     "kept": [names[c] for c in keep]}
    np.savez_compressed(OUT / f"{name}.npz", pred=p, gt=g, kept=keep)
    for i in range(min(4, len(p))):
        Image.fromarray((n01(p[i]) * 255).astype(np.uint8)).save(OUT / "png" / f"{name}_s{i}_pred.png")
    print(f"{name:10s} keep={'+'.join(names[c] for c in keep):12s} "
          f"SSIM {np.mean(ss):.4f} ± {np.std(ss):.4f}")

json.dump(results, open(OUT / "summary.json", "w"), indent=2)
print("saved:", OUT)

## 9. Full test-set inference (NFE=100)

In [ ]:
import pickle

dm = FlowMatchingDataModule(config)
dm.setup(stage="validate")
loader = dm.val_dataloader()  # shuffle=False -> order matches test_idx
print("test slices:", len(loader.dataset))

preds, gts, masks = [], [], []
with torch.no_grad():
    for batch in tqdm(loader, desc=f"inference NFE={NFE}"):
        out = sample_batch(model.model, SOLVER, batch, torch.device(DEVICE),
                           class_conditioning=bool(MC.get("with_conditioning", False)),
                           mask_conditioning=bool(MC.get("mask_conditioning", False)))
        preds.extend(out.detach().float().cpu().numpy()[:, 0])
        gts.extend(batch["images"].numpy()[:, 0])
        masks.extend(batch["masks"].numpy())
print(len(preds), "predictions")

CFG.out_dir.mkdir(parents=True, exist_ok=True)
pickle.dump({"test": [{"image": p[None, ...]} for p in preds]},
            open(CFG.out_dir / f"samples_nfe{NFE}.pkl", "wb"))

def _n01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

rows = list(test_idx.itertuples(index=False))[:len(preds)]
test_list = [{"metadata": {"pid": r.pid, "z": int(r.z), "file": r.file},
              "class": int(getattr(r, "has_tumour", 0)),
              "image": gts[i][None, ...], "mask": masks[i]} for i, r in enumerate(rows)]
opt_nfe = NFE
ss = [float(skimage_ssim(_n01(preds[i]), _n01(gts[i]), data_range=1.0)) for i in range(len(preds))]
print(f"NFE={NFE}  SSIM {np.mean(ss):.4f} ± {np.std(ss):.4f}")

## 10. 3D TIFF volume export

In [ ]:
# Assemble per-slice predictions into GT/pred 3D TIFF stacks (>= 20 patients).
import tifffile

PRED_ROOT = CFG.out_dir / "predictions_3d"
PRED_ROOT.mkdir(parents=True, exist_ok=True)
MIN_PATIENTS = 20

by_pid = defaultdict(list)
for i in range(len(test_list)):
    by_pid[test_list[i]["metadata"]["pid"]].append(i)
pids_sorted = sorted(by_pid.keys(), key=lambda p: -len(by_pid[p]))

exported = 0
for pid in pids_sorted:
    sl = sorted(by_pid[pid], key=lambda i: test_list[i]["metadata"]["z"])
    if len(sl) < 5:
        continue
    gt_stack = np.stack([_n01(np.asarray(test_list[i]["image"])[0]) for i in sl], axis=0)
    pred_stack = np.stack([_n01(preds[i]) for i in sl], axis=0)
    pdir = PRED_ROOT / pid
    pdir.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(pdir / "t1ce_gt.tif", (gt_stack * 255).astype(np.uint8))
    tifffile.imwrite(pdir / "t1ce_motfm_pred.tif", (pred_stack * 255).astype(np.uint8))
    exported += 1
    print(f"  [{exported:2d}] {pid}: {len(sl)} slices")
    if exported >= MIN_PATIENTS:
        break
print(f"exported {exported} volumes → {PRED_ROOT}")

In [ ]:
# Stackwise comparison figure for a few exported volumes.
import tifffile
vol_dirs = sorted(d for d in (CFG.out_dir / "predictions_3d").iterdir() if d.is_dir())[:3]
fig, axes = plt.subplots(len(vol_dirs), 6, figsize=(15, 2.6 * len(vol_dirs)))
if len(vol_dirs) == 1:
    axes = axes[None, :]
for r, vd in enumerate(vol_dirs):
    gt = tifffile.imread(vd / "t1ce_gt.tif")
    pred = tifffile.imread(vd / "t1ce_motfm_pred.tif")
    Z = gt.shape[0]
    for c, z in enumerate(np.linspace(Z // 5, 4 * Z // 5, 3).astype(int)):
        axes[r, c * 2].imshow(gt[z], cmap="gray"); axes[r, c * 2].set_title(f"GT z={z}", fontsize=8)
        axes[r, c * 2 + 1].imshow(pred[z], cmap="gray"); axes[r, c * 2 + 1].set_title(f"Pred z={z}", fontsize=8)
        axes[r, c * 2].axis("off"); axes[r, c * 2 + 1].axis("off")
    axes[r, 0].set_ylabel(vd.name, fontsize=8)
plt.suptitle("3D volumes — GT vs predicted T1CE", fontsize=11)
plt.tight_layout()
plt.savefig(CFG.out_dir / "volume_stackwise_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## 11. Full metrics — EVarΔ / Wasserstein / multimodality

In [ ]:
import pickle
NFE_EVAL = int(opt_nfe) if "opt_nfe" in dir() else 100
THRESH = 0.05

def _gt_image(s):
    g = np.asarray(s["image"])
    return g[0] if g.ndim == 3 else g

if "preds" not in dir():
    gen = pickle.load(open(CFG.out_dir / f"samples_nfe{NFE_EVAL}.pkl", "rb"))
    key = "test" if "test" in gen else next(iter(gen))
    preds = [np.asarray(s["image"], np.float32) for s in gen[key]]
    preds = [p[0] if p.ndim == 3 else p for p in preds]

M = min(len(preds), len(test_list))
pred01, gt01, t1n01, meta = [], [], [], []
for i in range(M):
    s = test_list[i]
    pred01.append(_n01(preds[i]))
    gt01.append(_n01(_gt_image(s)))
    t1n01.append(_n01(np.asarray(s["mask"])[0]))
    meta.append((s["metadata"]["pid"], s["metadata"]["z"], bool(s["class"])))

ssim_all = [float(skimage_ssim(pred01[i], gt01[i], data_range=1.0)) for i in range(M)]
psnr_all = [float(skimage_psnr(gt01[i], pred01[i], data_range=1.0)) for i in range(M)]
tum = [i for i in range(M) if meta[i][2]]
heal = [i for i in range(M) if not meta[i][2]]
ssim_tum = float(np.mean([ssim_all[i] for i in tum])) if tum else None
ssim_heal = float(np.mean([ssim_all[i] for i in heal])) if heal else None

by_pid = defaultdict(list)
for i in range(M):
    if meta[i][2]:
        by_pid[meta[i][0]].append(i)
evar_vals = []
for pid, sl in by_pid.items():
    sl = sorted(sl, key=lambda i: meta[i][1])
    dp, dg = [], []
    for i in sl:
        enh = (gt01[i] - t1n01[i]) > THRESH
        if enh.sum() < 10:
            continue
        dp.append(float((pred01[i] - t1n01[i])[enh].mean()))
        dg.append(float((gt01[i] - t1n01[i])[enh].mean()))
    if len(dg) >= 2:
        vg = float(np.var(dg, ddof=1))
        if vg > 1e-8:
            evar_vals.append(float(np.var(dp, ddof=1)) / vg)
evar_mean = float(np.mean(evar_vals)) if evar_vals else None
evar_std = float(np.std(evar_vals)) if evar_vals else None

ap, ag = [], []
for i in range(M):
    if not meta[i][2]:
        continue
    enh = (gt01[i] - t1n01[i]) > THRESH
    if enh.sum() < 5:
        continue
    ap.extend((pred01[i] - t1n01[i])[enh].tolist())
    ag.extend((gt01[i] - t1n01[i])[enh].tolist())
if ag:
    rng = np.random.default_rng(42)
    N = min(len(ag), 100_000)
    d_W = float(wasserstein_distance(rng.choice(ag, N, replace=False),
                                     rng.choice(ap, N, replace=False)))
else:
    d_W = None

try:
    train_pids = set(splits_df.loc[splits_df["split"] == "train", "patient_id"])
    train_m = idx[idx["pid"].isin(train_pids) & (idx["has_tumour"] == True)]
    multi_L1 = prove_multimodality(train_m, CFG.tensors_dir, n_pairs=200).get("matched_L1_mean")
except Exception as e:
    multi_L1 = None
    print("multimodality skipped:", e)

print(f"MOTFM — NFE={NFE_EVAL} — {M} slices — {len(by_pid)} tumour patients")
print(f"SSIM (all)     : {np.mean(ssim_all):.4f} ± {np.std(ssim_all):.4f}")
print(f"SSIM (tumour)  : {ssim_tum:.4f}" if ssim_tum else "SSIM (tumour): n/a")
print(f"SSIM (healthy) : {ssim_heal:.4f}" if ssim_heal else "SSIM (healthy): n/a")
print(f"PSNR (all)     : {np.mean(psnr_all):.2f} dB")
print(f"EVarΔ          : {evar_mean:.4f} ± {evar_std:.4f}  (n={len(evar_vals)})" if evar_mean else "EVarΔ: n/a")
print(f"Wasserstein    : {d_W:.4f}  ({len(ag):,} voxels)" if d_W else "d_W: n/a")
print(f"Multimodality  : {multi_L1:.4f}" if multi_L1 else "Multimodality: n/a")

summary = {"model": "MOTFM", "nfe": NFE_EVAL, "n_slices": M, "n_tumour_patients": len(by_pid),
           "ssim": float(np.mean(ssim_all)), "ssim_tumour": ssim_tum, "ssim_healthy": ssim_heal,
           "psnr": float(np.mean(psnr_all)), "evar_mean": evar_mean, "evar_std": evar_std,
           "evar_n": len(evar_vals), "d_W": d_W, "multi_L1_mean": multi_L1}
json.dump(summary, open(CFG.out_dir / "metrics_summary.json", "w"), indent=2, default=str)
print("saved metrics_summary.json")

## 12. Clean metrics table

In [ ]:
# Comparison table (Yazdani et al., MICCAI 2025 style). Baselines from prior runs.
rows = [
    ("Identity (T1n)", 0.765, 0.721, 0.789, 17.83, 0.562, None),
    ("U-Net", 0.873, 0.872, 0.874, 25.97, 0.185, None),
    ("U-Net + LPIPS", 0.872, 0.856, 0.879, 26.32, 0.174, None),
    ("MOTFM (ours)", round(float(np.mean(ssim_all)), 3),
     round(ssim_tum, 3) if ssim_tum else None,
     round(ssim_heal, 3) if ssim_heal else None,
     round(float(np.mean(psnr_all)), 2), None,
     round(d_W, 3) if d_W else None),
]
df = pd.DataFrame(rows, columns=["Model", "SSIM (all)", "SSIM (tum)", "SSIM (heal)",
                                 "PSNR (dB)", "MAE", "d_W"])
print(df.to_string(index=False))
df.to_csv(CFG.out_dir / "clean_metrics_table.csv", index=False)
print("saved clean_metrics_table.csv")

## 13. Package results

In [ ]:
# Zip metrics, tables, figures and the 3D TIFF volumes for download.
zp = CFG.persist_base / "synT1CE_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ["metrics_summary.json", "clean_metrics_table.csv",
                 "volume_stackwise_comparison.png"]:
        p = CFG.out_dir / name
        if p.exists():
            zf.write(p, name)
    for tif in (CFG.out_dir / "predictions_3d").rglob("*.tif"):
        zf.write(tif, f"predictions_3d/{tif.parent.name}/{tif.name}")
print(f"{zp}  ({zp.stat().st_size / 1e6:.1f} MB)")